# 365 Probabilidades - Dia #123
## Qual a probabilidade de a IA que você usa ter sido feita por uma mulher?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-14
**Ferramenta:** Python
**Decisão analisada:** Quem está fazendo a IA, e o que isso muda no que ela aprende?
**Hashtag:** #365Probabilidades #Dia123

---

### 📖 A História

[Uma linha da Ana: o livro sai hoje, ela é uma das coautoras. Sem entorno.]

A pergunta do dia vem daí. Não "quantas mulheres trabalham com IA", que é a pergunta
que todo relatório responde. A outra: quando você abre um assistente, um modelo, um
sistema de recomendação, qual a chance de ter uma mulher entre quem o fez?

---

### 📚 O Conceito

A IA aprende com dados, e os dados são escritos por pessoas. Caliskan, Bryson &
Narayanan (2017, *Science*) mediram isso: num vetor de palavras treinado em 840 bilhões
de palavras da internet, a distância entre o nome de uma profissão e as palavras de
gênero acompanha a fração real de mulheres naquela profissão, com correlação de 0,90 em
50 ocupações. A máquina não inventa o viés. Ela mede quem está nos dados.

Por isso a pergunta de quem faz a IA não é só de justiça. É de dado.

---

### 🧮 O Modelo

Dia de **probabilidade simples**: a pergunta vem primeiro, e o dado é contagem.

**Dados:**
- INEP, *Censo da Educação Superior*, microdados de **2022 e 2024**, tabela de cursos
  (`MICRODADOS_CADASTRO_CURSOS`). Ingressantes, matrículas e concluintes por sexo, para
  as onze áreas gerais da classificação CINE Brasil e para cada curso da área
  "Computação e Tecnologias da Informação e Comunicação (TIC)". Somados por nós a partir
  de 573.019 (2022) e 720.349 (2024) linhas; o resumo está em
  `inep_censo_superior_computacao_por_sexo.csv`, ao lado deste notebook.
- Stathoulopoulos, K. & Mateos-Garcia, J. (2019). *Gender Diversity in AI Research.*
  Nesta. 74.407 artigos de IA no arXiv até 2018; gênero inferido pelo nome (Gender API,
  92,1% de acerto declarado, pior em nomes asiáticos). **13,8%** dos autores são
  mulheres; **25,4%** dos artigos têm ao menos uma coautora. Relatório, não artigo
  revisado por pares: entra com a exceção declarada no corpo do texto.

**Fonte de conceito:**
- Caliskan, A., Bryson, J. J. & Narayanan, A. (2017). *Semantics derived automatically
  from language corpora contain human-like biases.* Science 356(6334), 183-186.
  ρ = 0,90 (p < 10⁻¹⁸) entre o viés de gênero do nome da ocupação no vetor e a fração de
  mulheres na ocupação (BLS 2015, 50 ocupações).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def beta_j(x, n):
    # posterior de Jeffreys para uma proporcao
    return stats.beta(x + 0.5, n - x + 0.5)

print("✅ Bibliotecas carregadas")

In [ ]:
# --- OS DADOS ---
ARQ = "inep_censo_superior_computacao_por_sexo.csv"
CANDIDATOS = [ARQ, os.path.join("dados", ARQ), os.path.join("..", "dados", ARQ),
              os.path.join("..", "dias-091-120", ARQ)]
caminho = next((c for c in CANDIDATOS if os.path.exists(c)), None)
if caminho is None:
    raise FileNotFoundError(f"Salve {ARQ} na mesma pasta deste notebook (pasta atual: {os.getcwd()}).")
d = pd.read_csv(caminho)
AREA_COMP = "Computação e Tecnologias da Informação e Comunicação (TIC)"
TODOS = "(todos os cursos da área)"

areas = d[d["NO_CINE_ROTULO"] == TODOS].set_index(["NU_ANO_CENSO", "NO_CINE_AREA_GERAL"])
cursos = d[(d["NO_CINE_ROTULO"] != TODOS) & (d["NO_CINE_AREA_GERAL"] == AREA_COMP)]

# Conferencias: a soma dos cursos de computacao fecha com a linha da area, nos dois anos
for ano in (2022, 2024):
    soma = cursos[cursos["NU_ANO_CENSO"] == ano][["QT_ING", "QT_ING_FEM", "QT_CONC", "QT_CONC_FEM"]].sum()
    linha = areas.loc[(ano, AREA_COMP), ["QT_ING", "QT_ING_FEM", "QT_CONC", "QT_CONC_FEM"]]
    assert (soma.values == linha.values).all(), f"soma dos cursos não fecha em {ano}"

# arXiv (Nesta 2019)
ARXIV_AUTORAS = 0.138       # fracao de autores de artigos de IA que sao mulheres
ARXIV_ART_COM_MULHER = 0.254  # fracao de artigos com ao menos uma coautora
ARXIV_N_ARTIGOS = 74_407

# Caliskan 2017
RHO_CALISKAN, N_OCUP = 0.90, 50

print("=" * 72)
print("  OS DADOS: QUEM ENTRA, QUEM FICA, QUEM SE FORMA EM COMPUTAÇÃO")
print("=" * 72)
for ano in (2022, 2024):
    c = areas.loc[(ano, AREA_COMP)]
    b = areas.loc[ano].sum()
    print(f"\n  {ano} · Computação e TIC")
    print(f"    ingressantes  {mil(c['QT_ING']):>10}   mulheres {mil(c['QT_ING_FEM']):>8}   {pct(c['QT_ING_FEM'] / c['QT_ING'])}")
    print(f"    matrículas    {mil(c['QT_MAT']):>10}   mulheres {mil(c['QT_MAT_FEM']):>8}   {pct(c['QT_MAT_FEM'] / c['QT_MAT'])}")
    print(f"    concluintes   {mil(c['QT_CONC']):>10}   mulheres {mil(c['QT_CONC_FEM']):>8}   {pct(c['QT_CONC_FEM'] / c['QT_CONC'])}")
    print(f"  {ano} · Brasil, todas as áreas: ingressantes {pct(b['QT_ING_FEM'] / b['QT_ING'])} mulheres, "
          f"concluintes {pct(b['QT_CONC_FEM'] / b['QT_CONC'])}")
print(f"\n  Conferência: a soma dos cursos de computação fecha com a linha da área nos dois anos.")
print(f"\n  arXiv (Nesta 2019): {pct(ARXIV_AUTORAS)} dos autores de IA são mulheres; "
      f"{pct(ARXIV_ART_COM_MULHER)} dos artigos têm ao menos uma.")
print(f"\n  ×0,80: não se aplica. Contagem administrativa (Censo) e contagem de autores.")
print("=" * 72)

In [ ]:
# --- O MODELO ---

# ── CAMADA 1 · a proporcao, com intervalo ──
c24 = areas.loc[(2024, AREA_COMP)]
post_ing = beta_j(c24["QT_ING_FEM"], c24["QT_ING"])
post_conc = beta_j(c24["QT_CONC_FEM"], c24["QT_CONC"])
print("=" * 72)
print("  CAMADA 1 · UMA EM CINCO ENTRA, UMA EM CINCO SE FORMA (2024)")
print("=" * 72)
for nome, p in [("ingressantes", post_ing), ("concluintes", post_conc)]:
    lo, hi = p.interval(0.95)
    print(f"  {nome:<14}{pct(p.mean(), 2):>8}   intervalo de credibilidade de 95% [{pct(lo, 2)}; {pct(hi, 2)}]")
print(f"  Com centenas de milhares de registros, o intervalo é da largura de um fio.")
print(f"  O Censo é contagem, não amostra: a incerteza aqui é de registro, não amostral.")

# ── CAMADA 2 · o funil e a mudanca 2022 -> 2024 ──
print("\n" + "=" * 72)
print("  CAMADA 2 · O FUNIL, E O QUE MUDOU DE 2022 PARA 2024")
print("=" * 72)
N_SIM = 200_000
res = {}
for ano in (2022, 2024):
    c = areas.loc[(ano, AREA_COMP)]
    res[ano] = dict(ing=beta_j(c["QT_ING_FEM"], c["QT_ING"]).rvs(N_SIM, random_state=ger),
                    conc=beta_j(c["QT_CONC_FEM"], c["QT_CONC"]).rvs(N_SIM, random_state=ger))
for ano in (2022, 2024):
    gap = res[ano]["ing"] - res[ano]["conc"]
    print(f"  {ano}: entra {pct(res[ano]['ing'].mean())}, forma {pct(res[ano]['conc'].mean())}, "
          f"diferença {vir(gap.mean() * 100)} pp [{vir(np.percentile(gap, 2.5) * 100)}; {vir(np.percentile(gap, 97.5) * 100)}]")
d_ing = res[2024]["ing"] - res[2022]["ing"]; d_conc = res[2024]["conc"] - res[2022]["conc"]
print(f"\n  2024 menos 2022, ingressantes: {vir(d_ing.mean() * 100, 2)} pp [{vir(np.percentile(d_ing, 2.5) * 100, 2)}; {vir(np.percentile(d_ing, 97.5) * 100, 2)}]")
print(f"  2024 menos 2022, concluintes:  {vir(d_conc.mean() * 100, 2)} pp [{vir(np.percentile(d_conc, 2.5) * 100, 2)}; {vir(np.percentile(d_conc, 97.5) * 100, 2)}]")
print(f"  A fração de mulheres entre concluintes subiu quase 4 pontos em dois anos; o número")
print(f"  de concluintes da área subiu de {mil(areas.loc[(2022, AREA_COMP), 'QT_CONC'])} para "
      f"{mil(areas.loc[(2024, AREA_COMP), 'QT_CONC'])}. Dois anos não fazem tendência; registro.")

# ── CAMADA 3 · curso a curso ──
print("\n" + "=" * 72)
print("  CAMADA 3 · CURSO A CURSO (2024, os oito maiores)")
print("=" * 72)
c24c = cursos[cursos["NU_ANO_CENSO"] == 2024].sort_values("QT_ING", ascending=False).head(8).copy()
c24c["p_ing"] = c24c["QT_ING_FEM"] / c24c["QT_ING"]; c24c["p_conc"] = c24c["QT_CONC_FEM"] / c24c["QT_CONC"]
print(f"\n  {'curso':<40}{'ingressantes':>13}{'mulheres':>10}{'formam':>9}")
for _, r in c24c.iterrows():
    nome = r["NO_CINE_ROTULO"][:38]
    print(f"  {nome:<40}{mil(r['QT_ING']):>13}{pct(r['p_ing']):>10}{pct(r['p_conc']):>9}")
cc = c24c.set_index("NO_CINE_ROTULO")
print(f"\n  Ciência da Computação, o curso de base da IA: {pct(cc.loc['Ciência da computação', 'p_ing'])} entram, "
      f"{pct(cc.loc['Ciência da computação', 'p_conc'])} se formam.")
print(f"  Ciência de Dados, o curso que mais alimenta a IA hoje: {pct(cc.loc['Ciência de dados', 'p_ing'])} entram, "
      f"{pct(cc.loc['Ciência de dados', 'p_conc'])} se formam.")

# ── CAMADA 4 · a equipe: a pergunta do titulo ──
# "Feita por uma mulher" tem duas leituras. A estrita: a pessoa que assina e mulher.
# A larga: ha ao menos uma mulher na equipe. A segunda depende do tamanho da equipe.
print("\n" + "=" * 72)
print("  CAMADA 4 · A PERGUNTA DO TÍTULO, NAS DUAS LEITURAS")
print("=" * 72)
p_arxiv = ARXIV_AUTORAS
p_br = post_conc.mean()
print(f"\n  Leitura estrita: quem assina é mulher.")
print(f"    pesquisa de IA no mundo (arXiv): {pct(p_arxiv)}")
print(f"    quem se forma em computação no Brasil: {pct(p_br)}")
print(f"\n  Leitura larga: ao menos uma mulher na equipe, se cada vaga fosse sorteada ao acaso.")
print(f"  {'equipe':>8}{'arXiv (13,8%)':>16}{'Brasil (19,2%)':>16}")
tam = [1, 2, 3, 5, 10, 20]
for n in tam:
    print(f"  {n:>8}{pct(1 - (1 - p_arxiv) ** n):>16}{pct(1 - (1 - p_br) ** n):>16}")
# O arXiv permite testar a hipotese do sorteio: 25,4% dos artigos tem ao menos uma mulher.
k_star = np.log(1 - ARXIV_ART_COM_MULHER) / np.log(1 - p_arxiv)
print(f"\n  Teste com o dado que existe: {pct(ARXIV_ART_COM_MULHER)} dos artigos têm ao menos uma coautora.")
print(f"  Sob sorteio independente, isso corresponde a equipes de {vir(k_star, 2)} autores.")
print(f"  Artigos de IA costumam ter mais autores que isso. Se tiverem 3, o sorteio previa "
      f"{pct(1 - (1 - p_arxiv) ** 3)}; com 4, {pct(1 - (1 - p_arxiv) ** 4)}.")
print(f"  O observado fica abaixo: as mulheres não estão espalhadas pelas equipes, estão")
print(f"  concentradas em algumas. O tamanho médio das equipes não está no relatório, então")
print(f"  isto é leitura, declarada, não estimativa.")

# ── CAMADA 5 · o que a maquina aprende de quem esta nos dados ──
z = np.arctanh(RHO_CALISKAN); se = 1 / np.sqrt(N_OCUP - 3)
lo_r, hi_r = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
print("\n" + "=" * 72)
print("  CAMADA 5 · O QUE A MÁQUINA APRENDE DE QUEM ESTÁ NOS DADOS")
print("=" * 72)
print(f"\n  Caliskan et al. 2017: ρ = {vir(RHO_CALISKAN, 2)} entre o viés de gênero do nome da ocupação")
print(f"  no vetor de palavras e a fração de mulheres na ocupação (50 ocupações).")
print(f"  Intervalo de 95% por z de Fisher: [{vir(lo_r, 2)}; {vir(hi_r, 2)}]. R² = {pct(RHO_CALISKAN ** 2, 0)}.")
print(f"  A máquina reproduz a composição do mundo. Mudar quem faz e mudar o que ela aprende")
print(f"  são a mesma coisa, com atraso de uma geração de dados.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · o funil, computacao contra o Brasil ══
fig1, ax1 = plt.subplots(figsize=(11, 6))
etapas = ["ingressantes", "matrículas", "concluintes"]
c = areas.loc[(2024, AREA_COMP)]; b = areas.loc[2024].sum()
comp_v = [c["QT_ING_FEM"] / c["QT_ING"], c["QT_MAT_FEM"] / c["QT_MAT"], c["QT_CONC_FEM"] / c["QT_CONC"]]
br_v = [b["QT_ING_FEM"] / b["QT_ING"], b["QT_MAT_FEM"] / b["QT_MAT"], b["QT_CONC_FEM"] / b["QT_CONC"]]
x = np.arange(3)
ax1.bar(x - 0.2, np.array(br_v) * 100, width=0.38, color=CINZA, alpha=0.8, label="todas as áreas")
ax1.bar(x + 0.2, np.array(comp_v) * 100, width=0.38, color=VERMELHO, alpha=0.9, label="computação e TIC")
for i in range(3):
    ax1.text(x[i] - 0.2, br_v[i] * 100 + 1.2, pct(br_v[i]), ha="center", fontsize=12, fontweight="bold", color=CINZA)
    ax1.text(x[i] + 0.2, comp_v[i] * 100 + 1.2, pct(comp_v[i]), ha="center", fontsize=12, fontweight="bold", color=VERMELHO)
ax1.axhline(50, color="#333333", lw=1, ls="--"); ax1.text(2.45, 51, "metade", fontsize=10, ha="right", color="#333333")
ax1.set_xticks(x); ax1.set_xticklabels(etapas, fontsize=12)
ax1.set_ylim(0, 72); ax1.set_ylabel("mulheres (%)")
ax1.legend(fontsize=10, loc="upper right")
ax1.set_title("Mulheres são 60% de quem se forma no Brasil, e 19% de quem se forma em computação\n"
              "Censo da Educação Superior 2024, microdados por curso somados pela área CINE", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: INEP, Censo da Educação Superior 2024 · contagem administrativa  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-123-grafico-01-o-funil.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · curso a curso ══
fig2, ax2 = plt.subplots(figsize=(11.5, 6.6))
cc2 = c24c.sort_values("p_ing")
y = np.arange(len(cc2))
ax2.barh(y - 0.19, cc2["p_ing"] * 100, height=0.36, color=DOURADO, alpha=0.9, label="entram")
ax2.barh(y + 0.19, cc2["p_conc"] * 100, height=0.36, color=VERMELHO, alpha=0.9, label="se formam")
for i, (_, r) in enumerate(cc2.iterrows()):
    ax2.text(r["p_ing"] * 100 + 0.5, i - 0.19, pct(r["p_ing"]), va="center", fontsize=10, color="#8a6f1f")
    ax2.text(r["p_conc"] * 100 + 0.5, i + 0.19, pct(r["p_conc"]), va="center", fontsize=10, color=VERMELHO)
ax2.set_yticks(y); ax2.set_yticklabels([n[:34] for n in cc2["NO_CINE_ROTULO"]], fontsize=10)
ax2.set_xlim(0, 40); ax2.set_xlabel("mulheres (%)")
ax2.legend(fontsize=10, loc="lower right")
ax2.set_title("Curso a curso: Ciência da Computação em 17% na entrada e 13% na formatura\n"
              "os oito maiores cursos da área em 2024, por ingressantes", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: INEP, Censo da Educação Superior 2024  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-123-grafico-02-curso-a-curso.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · 2022 -> 2024 ══
fig3, ax3 = plt.subplots(figsize=(10, 5.8))
for nome, cor, chave in [("ingressantes", DOURADO, "ing"), ("concluintes", VERMELHO, "conc")]:
    vals = [res[a][chave].mean() * 100 for a in (2022, 2024)]
    ax3.plot([2022, 2024], vals, marker="o", ms=10, lw=2.8, color=cor, label=nome)
    for a, v in zip((2022, 2024), vals):
        ax3.text(a, v + 0.5, pct(v / 100), ha="center", fontsize=11, fontweight="bold", color=cor)
ax3.set_xticks([2022, 2024]); ax3.set_xlim(2021.5, 2024.5); ax3.set_ylim(12, 25)
ax3.set_ylabel("mulheres em computação e TIC (%)")
ax3.legend(fontsize=10, loc="upper left")
ax3.set_title(f"Entre 2022 e 2024, a fração de mulheres entre concluintes subiu {vir(d_conc.mean() * 100)} pontos\n"
              "dois anos não fazem tendência; é o registro mais recente", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: INEP, microdados 2022 e 2024 · intervalos de credibilidade menores que o marcador  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-123-grafico-03-dois-anos.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a equipe ══
fig4, ax4 = plt.subplots(figsize=(11, 6))
ns = np.arange(1, 21)
ax4.plot(ns, (1 - (1 - p_arxiv) ** ns) * 100, marker="o", lw=2.6, color=VERMELHO, label=f"pesquisa de IA no mundo, {pct(p_arxiv)} de autoras")
ax4.plot(ns, (1 - (1 - p_br) ** ns) * 100, marker="o", lw=2.6, color=VERDE, label=f"formandos em computação no Brasil, {pct(p_br)} de mulheres")
ax4.axhline(ARXIV_ART_COM_MULHER * 100, color=DOURADO, lw=2, ls="--")
ax4.text(20, ARXIV_ART_COM_MULHER * 100 + 1.5, f"observado no arXiv: {pct(ARXIV_ART_COM_MULHER)} dos artigos têm ao menos uma mulher",
         ha="right", fontsize=10, color="#8a6f1f")
ax4.set_xlabel("tamanho da equipe (pessoas)"); ax4.set_ylabel("chance de haver ao menos uma mulher (%)")
ax4.set_xticks([1, 2, 3, 5, 10, 15, 20]); ax4.set_ylim(0, 100)
ax4.legend(fontsize=10, loc="lower right")
ax4.set_title("Se as vagas fossem sorteadas, uma equipe de cinco teria uma mulher em 52% a 66% das vezes\n"
              "o observado nos artigos fica abaixo da curva: as mulheres estão concentradas, não espalhadas", fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Conta própria: 1 − (1 − p)ⁿ sob sorteio independente, hipótese declarada · Nesta 2019 e INEP 2024  |  #365Probabilidades",
            ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-123-grafico-04-a-equipe.png", dpi=150, bbox_inches="tight"); plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Se a IA que você usa tem uma assinatura, a chance de ser de uma mulher é de 13,8% no
mundo da pesquisa, e de 19,2% entre quem se forma em computação no Brasil.**

Uma em cinco entra. Uma em cinco se forma. No resto do ensino superior brasileiro, as
mulheres são seis em dez. Em Ciência da Computação, o curso de base da IA, 17% entram e
13% se formam. Em Ciência de Dados, 28% e 25%.

A notícia que o registro traz: de 2022 para 2024, a fração de mulheres entre concluintes
de computação subiu de 15,3% para 19,2%. Dois anos não fazem tendência. Mas é a direção
certa, e é o dado mais recente que existe.

Se as vagas das equipes fossem sorteadas, uma equipe de cinco pessoas teria ao menos uma
mulher em mais da metade das vezes. O arXiv mostra que não é assim: só um em cada quatro
artigos de IA tem uma coautora. As mulheres não estão espalhadas pelas equipes. Estão
concentradas em algumas.

E a máquina aprende com isso. A correlação entre o que o vetor de palavras acha de uma
profissão e a fração de mulheres nela é de 0,90. A IA não inventa o viés. Ela mede quem
está nos dados. Mudar quem faz e mudar o que ela aprende são a mesma coisa.

*Quantas mulheres havia na sala em que a última ferramenta que você usou foi decidida?*

---

### ⚠️ Limitações do Modelo

- O Censo conta ingressantes e concluintes de cursos de computação, não quem trabalha
  com IA. É o melhor registro brasileiro de quem entra na porta, não de quem chega à sala.
- O arXiv de Nesta é relatório, não artigo revisado por pares, e infere gênero pelo nome
  (92,1% de acerto declarado, pior em nomes asiáticos). Entra com a exceção declarada
  no corpo do texto. Os dados vão até 2018.
- A leitura larga da pergunta (ao menos uma mulher na equipe) supõe sorteio
  independente, que o próprio arXiv mostra não valer. A curva é régua, não estimativa.
- O tamanho médio das equipes de artigos de IA não está no relatório; a conclusão de
  concentração é leitura, declarada como tal.
- 2022 e 2024 são dois pontos. A subida entre eles é registro, não tendência.
- Caliskan et al. 2017 mediram vetores de palavras de 2014; os modelos de hoje são
  outros. A correlação entra como conceito, não como medida dos modelos atuais.
- O Censo tem 2023 publicado; não foi incluído. Se entrar, a série ganha o ponto do meio.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*